# Evaluating RAG: Faithfulness, Relevancy, Correctness

Every earlier episode trusted the printed answer by eye. In production you need to actually measure quality — LlamaIndex ships its own evaluators for exactly this, built into the same framework you're already using:

- **Faithfulness** — is the answer actually supported by the retrieved context, or did the LLM make something up?
- **Relevancy** — does the answer actually address the query and the retrieved context?
- **Correctness** — does the answer match a known-correct reference answer?


**Step 1 — Setup.** Configure logging, load API keys, and set the default LLM/embedding model — plus a second, separate LLM (`eval_llm`) whose only job is to judge answer quality.


In [1]:
import logging

from dotenv import load_dotenv
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

# Quiet down noisy INFO-level logs from the HTTP client, LlamaIndex, and the
# OpenAI SDK so only real problems show up below.
for noisy_logger in ("httpx", "llama_index", "openai"):
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)

# Loads keys like OPENAI_API_KEY from .env into os.environ.
load_dotenv()

# These are the defaults every index/query engine in this notebook will use.
Settings.llm = OpenAI(model="gpt-4.1-nano")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# The evaluator LLM judges quality, so it needs to be reliable — gpt-4o-mini
# instead of the project-wide gpt-4.1-nano default, same reasoning as earlier
# episodes that need dependable structured judgment.
eval_llm = OpenAI(model="gpt-4o-mini")

**Step 2 — Build the pipeline and get one answer.** Same load → index → query flow as every earlier episode. This one answer is what we'll actually evaluate below.


In [2]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

# The same load -> chunk -> embed -> index pipeline used throughout the series.
documents = SimpleDirectoryReader("data/sample_docs").load_data()
index = VectorStoreIndex.from_documents(documents)

query = "What is Naruto's signature technique?"
response = index.as_query_engine().query(query)  # this Response object is what gets evaluated below
print(f"Q: {query}\nA: {response}")

Q: What is Naruto's signature technique?
A: Naruto's signature technique is the Rasengan, a swirling ball of concentrated chakra.


**Step 3 — Score it without ground truth.** `FaithfulnessEvaluator` checks whether the answer is actually supported by the nodes that were retrieved; `RelevancyEvaluator` checks whether the answer and the retrieved context actually address the query. Neither needs a "correct answer" to compare against.


In [3]:
from llama_index.core.evaluation import FaithfulnessEvaluator, RelevancyEvaluator

# evaluate_response() re-reads the response's own source_nodes internally — no
# separate context needs to be passed in.
faithfulness_result = FaithfulnessEvaluator(llm=eval_llm).evaluate_response(query=query, response=response)
relevancy_result = RelevancyEvaluator(llm=eval_llm).evaluate_response(query=query, response=response)

# passing is a boolean verdict; score is the underlying numeric judgment (0-1 here).
print(f"Faithfulness: passing={faithfulness_result.passing}, score={faithfulness_result.score}")
print(f"Relevancy:    passing={relevancy_result.passing}, score={relevancy_result.score}")

Faithfulness: passing=True, score=1.0
Relevancy:    passing=True, score=1.0


**Step 4 — Score it against a known-correct answer.** `CorrectnessEvaluator` is different from the two above: it needs a human-written `reference` answer to compare against, which makes it useful for a curated test set rather than arbitrary live traffic.


In [4]:
from llama_index.core.evaluation import CorrectnessEvaluator

# reference is the ground-truth answer a human would accept — CorrectnessEvaluator
# scores how well the model's response matches it (score here is 0-5, not 0-1).
correctness_result = CorrectnessEvaluator(llm=eval_llm).evaluate(
    query=query,
    response=str(response),
    reference="The Rasengan, a swirling ball of concentrated chakra.",
)
print(f"Correctness: passing={correctness_result.passing}, score={correctness_result.score}")
print(f"Feedback: {correctness_result.feedback}")  # the judge LLM's written rationale

Correctness: passing=True, score=5.0
Feedback: The generated answer is fully relevant and correct, matching the reference answer precisely in both content and clarity.


**Step 5 — Prove Faithfulness actually catches hallucination.** Swap in a deliberately wrong answer (Goku's move, not Naruto's) while keeping the same retrieved context, then re-run Faithfulness to confirm it flags the mismatch instead of rubber-stamping every answer.


In [5]:
from llama_index.core.base.response.schema import Response
from llama_index.core.evaluation import FaithfulnessEvaluator

# Same retrieved context, deliberately wrong answer — this is Goku's technique,
# not Naruto's — to prove Faithfulness actually catches unsupported claims.
hallucinated_response = Response(
    response="Naruto's signature technique is the Kamehameha wave.",
    source_nodes=response.source_nodes,  # reuse the real context from Step 2
)
hallucination_check = FaithfulnessEvaluator(llm=eval_llm).evaluate_response(
    query=query, response=hallucinated_response
)
print(f"Hallucinated answer — Faithfulness: passing={hallucination_check.passing}")
print(f"Feedback: {hallucination_check.feedback}")

Hallucinated answer — Faithfulness: passing=False
Feedback: NO


### Summary

- Faithfulness and Relevancy need no ground truth — they only check the answer against its own retrieved context, so they can run on any query, even in production with no labeled data.
- Correctness needs a reference answer, which makes it the right tool for a curated test set (a handful of known Q&A pairs), not for evaluating arbitrary live traffic.
- These are LlamaIndex's own evaluators — a different tool from a dedicated evaluation framework like RAGAS, but useful precisely because it's already part of the framework you're building with, with no extra setup.
